手撕代码闭卷测试

In [ ]:
import torch
import math
import torch.functional as F

In [2]:
def layer_norm(x, gamma, beta, eps=1e-5):
    exp = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, keepdim=True, unbiased=False)
    output = (x - exp) / torch.sqrt(var + eps) * gamma + beta
    return output

In [ ]:
def rms_norm(x, gamma, eps=1e-5):
    output = x / torch.sqrt((x**2).mean(dim=-1, keepdim=True) + eps)
    return output * gamma

In [ ]:
def grpo_advantage(rewards, eps=1e-5):
    rewards = torch.tensor(rewards, dtype=torch.float32)
    mean = rewards.mean(dim=-1, keepdim=True)
    std = rewards.std(dim=-1, keepdim=True, unbiased=False)
    advantage = (rewards - mean) / (std + eps)
    return advantage

In [ ]:
def dpo_loss(policy_chosen_logps, policy_rejected_logps, ref_chosen_logps, ref_rejected_logps, beta=0.1):
    policy_logratio = policy_chosen_logps - policy_rejected_logps
    ref_logratio = ref_chosen_logps - ref_rejected_logps
    logits = beta * (policy_logratio - ref_logratio)
    loss = -F.logsigmoid(logits).mean()
    return loss

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = K.size(-1)
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask==0, float('-inf'))
    attn = torch.softmax(scores, dim=-1)
    output = attn @ V
    return output

In [ ]:
def softmax(x):
    x_max = x.max(dim=-1, keepdim=True).values
    exp_x = torch.exp(x - x_max)
    output = exp_x / exp_x.sum(dim=-1, keepdim=True)
    return output

In [ ]:

x.shape = [B, T, C]
num_heads = h
head_dim = C // h
q = q_proj(x)
k = k_proj(x)
v = v_proj(x)
q = q.view(B, T, h, head_dim).transpose(1, 2)
k = k.view(B, T, h, head_dim).transpose(1, 2)
v = v.view(B, T, h, head_dim).transpose(1, 2)
out = out.transpose(1, 2).contiguous().view(B, T, C)
casual_mask = torch.tril(torch.ones(T, T))

In [ ]:
def swiglu(x, w1, w2, w3):
    gate = F.silu(w1(x))
    value = w2(x)
    hidden = gate * value
    output = w3(hidden)
    return output